# ARC-D — FINAL publication runs (single source of truth)
Regenerates every accuracy/compute number in the paper. Protocol upgrades over the probe phase:
**cosine LR schedule** (fixes teacher instability), **20 epochs equal budget for all models** (2.5× previous — answers "KD undertrained" while keeping the equal-budget defense), **3 seeds** (mean±std), best-val checkpointing, val-only τ per seed, index-deterministic corruptions, paired TOST per seed.

### How to run (free-tier friendly, fully resumable)
- Set `DATASET` and `SEEDS_TO_RUN` in the config cell. One seed ≈ **35–45 min** on T4 (Freiburg slightly longer). Full battery = 2 datasets × 3 seeds ≈ **4–5 h total** — split across sessions/platforms freely.
- Everything is checkpointed per (dataset, model, seed): rerunning skips finished training; finished evals are skipped via the results CSV. Safe to crash and rerun.
- Set `USE_DRIVE=True` on Colab so checkpoints + `final_results.csv` survive sessions. On Kaggle, files persist in `/kaggle/working` automatically.
- Suggested split: session 1 = GSD seeds 0,1 · session 2 = GSD seed 2 + Freiburg seed 0 · session 3 = Freiburg seeds 1,2. (Or use Kaggle P100 for Freiburg.)
- After all runs: the **aggregation cell** reads the CSV and prints the paper tables (mean±std). It tolerates partial results.

Data splits and corruptions are **fixed across seeds** (val carve uses a fixed seed; occlusion is seeded by sample index) — only training stochasticity varies, so seeds are directly comparable.

In [ ]:
# ================= CONFIG =================
DATASET      = "freiburg"   # "gsd" or "freiburg"
SEEDS_TO_RUN = [0, 1, 2]    # run any subset; results accumulate in the CSV
EPOCHS       = 20
USE_DRIVE    = True         # Drive is the single source of truth (recommended)
# ==========================================

!pip -q install timm fvcore
import os, time, random, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2, timm
from fvcore.nn import FlopCountAnalysis

device = "cuda" if torch.cuda.is_available() else "cpu"
def detect_env():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "other"
BASE = "."
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/arcd_final"
elif detect_env() == "kaggle":
    BASE = "/kaggle/working"
CKPT_DIR = os.path.join(BASE, "final_ckpts"); os.makedirs(CKPT_DIR, exist_ok=True)
RESULTS  = os.path.join(BASE, "final_results.csv")
print("device:", device, "| base:", BASE, "| dataset:", DATASET, "| seeds:", SEEDS_TO_RUN)

In [ ]:
# ---------- dataset loading (both datasets, identical downstream protocol) ----------
import urllib.request, subprocess

def load_gsd():
    if not os.path.exists("GroceryStoreDataset"):
        subprocess.run(["git","clone","-q","https://github.com/marcusklasson/GroceryStoreDataset.git"], check=True)
    R="GroceryStoreDataset/dataset"
    def rd(t):
        out=[]
        with open(os.path.join(R,t)) as f:
            for line in f:
                line=line.strip()
                if not line: continue
                p=[x.strip() for x in line.split(",")]
                out.append((os.path.join(R,p[0]), int(p[1])))
        return out
    return rd("train.txt"), rd("val.txt"), rd("test.txt"), 81

def load_freiburg():
    if not os.path.exists("images"):
        url="http://aisdatasets.informatik.uni-freiburg.de/freiburg_groceries_dataset/freiburg_groceries_dataset.tar.gz"
        print("downloading (~1GB)..."); urllib.request.urlretrieve(url,"fg.tar.gz")
        subprocess.run(["tar","-xf","fg.tar.gz"], check=True)
    RAW="https://raw.githubusercontent.com/PhilJd/freiburg_groceries_dataset/master/splits"
    for f in ["train0.txt","test0.txt"]:
        if not os.path.exists(f): urllib.request.urlretrieve(f"{RAW}/{f}", f)
    def rd(t):
        seen=set(); out=[]
        with open(t) as fh:
            for line in fh:
                line=line.strip()
                if not line: continue
                rel,lab=line.rsplit(" ",1)
                if rel in seen: continue
                seen.add(rel); out.append((os.path.join("images",rel), int(lab)))
        return out
    tv, te = rd("train0.txt"), rd("test0.txt")
    rng=random.Random(0)                      # FIXED split seed: same val carve for every training seed
    byc={}
    for it in tv: byc.setdefault(it[1],[]).append(it)
    tr,va=[],[]
    for c,l in byc.items():
        l=l[:]; rng.shuffle(l); k=max(1,int(0.1*len(l)))
        va+=l[:k]; tr+=l[k:]
    return tr, va, te, 25

train_items, val_items, test_items, NUM_CLASSES = load_gsd() if DATASET=="gsd" else load_freiburg()
print(f"{DATASET}: train={len(train_items)} val={len(val_items)} test={len(test_items)} classes={NUM_CLASSES}")

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
def to_tensor_norm(img):
    t=torch.from_numpy(img).float().permute(2,0,1)/255.0
    for c in range(3): t[c]=(t[c]-MEAN[c])/STD[c]
    return t
def corrupt(img, kind, sev, idx=0):
    if kind=="none" or sev==0: return img
    if kind=="blur":
        k=[3,5,7,11,15][sev-1]; return cv2.GaussianBlur(img,(k,k),0)
    if kind=="jpeg":
        q=[40,30,20,12,7][sev-1]
        _,e=cv2.imencode(".jpg",cv2.cvtColor(img,cv2.COLOR_RGB2BGR),[cv2.IMWRITE_JPEG_QUALITY,q])
        return cv2.cvtColor(cv2.imdecode(e,cv2.IMREAD_COLOR),cv2.COLOR_BGR2RGB)
    if kind=="occlusion":
        s=[0.1,0.2,0.3,0.4,0.5][sev-1]; side=int(224*s)
        r=random.Random(10_000*sev+idx)
        x0=r.randint(0,224-side); y0=r.randint(0,224-side)
        img=img.copy(); img[y0:y0+side,x0:x0+side]=127; return img
    raise ValueError(kind)

class DS(Dataset):
    def __init__(self, items, train=False, kind="none", sev=0):
        self.items,self.train,self.kind,self.sev=items,train,kind,sev
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        path,label=self.items[i]
        img=np.array(Image.open(path).convert("RGB").resize((224,224)))
        if self.train and random.random()<0.5: img=img[:,::-1,:].copy()
        return to_tensor_norm(corrupt(img,self.kind,self.sev,idx=i)), label

In [ ]:
def build(name): return timm.create_model(name, pretrained=True, num_classes=NUM_CLASSES).to(device)

@torch.no_grad()
def quick_acc(model, items, bs=64):
    model.eval()
    dl=DataLoader(DS(items),batch_size=bs,num_workers=2,pin_memory=True)
    ok=n=0
    for x,y in dl:
        ok+=(model(x.to(device)).argmax(1).cpu()==y).sum().item(); n+=y.size(0)
    return ok/n

def train_model(model, name, seed, lr, teacher=None, T=4.0, alpha=0.7):
    tag=f"{DATASET}_{name}_seed{seed}"
    ckpt=os.path.join(CKPT_DIR, tag+".pt")            # best-val weights
    res =os.path.join(CKPT_DIR, tag+"_resume.pt")     # full per-epoch state
    if os.path.exists(ckpt) and not os.path.exists(res):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"  {name} seed{seed}: completed earlier, skipping"); return model.eval()
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    dl=DataLoader(DS(train_items,train=True),batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    best=-1.0; start_ep=0
    if os.path.exists(res):
        st=torch.load(res, map_location=device)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"]); start_ep=st["epoch"]+1; best=st["best"]
        print(f"  {name} seed{seed}: RESUMING at epoch {start_ep+1}/{EPOCHS} (best val {best*100:.1f}%)")
    for ep in range(start_ep, EPOCHS):
        model.train(); t0,run,n=time.time(),0.0,0
        for x,y in dl:
            x,y=x.to(device),y.to(device)
            logits=model(x)
            if teacher is None:
                loss=F.cross_entropy(logits,y)
            else:
                with torch.no_grad(): tl=teacher(x)
                soft=F.kl_div(F.log_softmax(logits/T,1),F.softmax(tl/T,1),reduction="batchmean")*(T*T)
                loss=alpha*soft+(1-alpha)*F.cross_entropy(logits,y)
            opt.zero_grad(); loss.backward(); opt.step()
            run+=loss.item()*x.size(0); n+=x.size(0)
        sched.step()
        va=quick_acc(model,val_items); star=""
        if va>best: best=va; torch.save(model.state_dict(),ckpt); star=" *"
        torch.save({"model":model.state_dict(),"opt":opt.state_dict(),
                    "sched":sched.state_dict(),"epoch":ep,"best":best}, res)
        print(f"  {name} s{seed} ep{ep+1}/{EPOCHS} loss={run/n:.4f} val={va*100:.1f}%{star} ({time.time()-t0:.0f}s)")
    if os.path.exists(res): os.remove(res)            # done marker: best.pt without resume file
    model.load_state_dict(torch.load(ckpt,map_location=device))
    print(f"  {name} seed{seed}: best val {best*100:.1f}%")
    return model.eval()

In [ ]:
@torch.no_grad()
def conf_correct(model, items, kind="none", sev=0, bs=64):
    model.eval()
    dl=DataLoader(DS(items,kind=kind,sev=sev),batch_size=bs,num_workers=2,pin_memory=True)
    C,K=[],[]
    for x,y in dl:
        p=torch.softmax(model(x.to(device)),1); c,pred=p.max(1)
        C.append(c.cpu().numpy()); K.append((pred.cpu()==y).numpy())
    return np.concatenate(C), np.concatenate(K)

def select_tau(stage1, heavy):
    c_v,k_v = conf_correct(stage1, val_items)
    _,kh_v  = conf_correct(heavy,  val_items)
    target=kh_v.mean()-0.01; best=None
    for t in np.concatenate([[0.0], np.linspace(0.5,0.999,60)]):
        keep=c_v>=t
        a=np.where(keep,k_v,kh_v).mean()
        g=G_CHEAP+(1-keep.mean())*G_HEAVY
        if a>=target and (best is None or g<best[1]): best=(t,g,a)
    return best[0]

def bootstrap_ci(diff, n=5000):
    idx=np.random.randint(0,len(diff),size=(n,len(diff)))
    b=diff[idx].mean(axis=1); return diff.mean(), np.percentile(b,5), np.percentile(b,95)

CONDS=[("clean","none",0),("jpeg_s2","jpeg",2),("blur_s2","blur",2),("occ_s1","occlusion",1)]
FIELDS=["dataset","seed","system","condition","acc","gmacs","routed","tau",
        "tost_mean","tost_lo","tost_hi"]

def done_keys():
    if not os.path.exists(RESULTS): return set()
    with open(RESULTS) as f:
        return {(r["dataset"],int(r["seed"]),r["system"],r["condition"]) for r in csv.DictReader(f)}

def append_rows(rows):
    new=not os.path.exists(RESULTS)
    with open(RESULTS,"a",newline="") as f:
        w=csv.DictWriter(f,fieldnames=FIELDS)
        if new: w.writeheader()
        for r in rows: w.writerow(r)

_g=timm.create_model("efficientnet_b0",pretrained=False,num_classes=NUM_CLASSES)
G_CHEAP=FlopCountAnalysis(_g,torch.randn(1,3,224,224)).total()/1e9
_g=timm.create_model("convnext_tiny",pretrained=False,num_classes=NUM_CLASSES)
G_HEAVY=FlopCountAnalysis(_g,torch.randn(1,3,224,224)).total()/1e9
del _g
print(f"GMACs cheap={G_CHEAP:.3f} heavy={G_HEAVY:.3f}")

In [ ]:
# ---------- MAIN LOOP: train + evaluate each seed, append to CSV ----------
existing = done_keys()
for seed in SEEDS_TO_RUN:
    if (DATASET,seed,"ARC-D","clean") in existing:
        print(f"seed {seed}: results already in CSV, skipping"); continue
    print(f"===== {DATASET} seed {seed} =====")
    heavy     = train_model(build("convnext_tiny"),   "heavy",     seed, 1e-4)
    cheap     = train_model(build("efficientnet_b0"), "cheap",     seed, 3e-4)
    distilled = train_model(build("efficientnet_b0"), "distilled", seed, 3e-4, teacher=heavy)

    tau_arc  = select_tau(cheap, heavy)
    tau_arcd = select_tau(distilled, heavy)
    print(f"  tau ARC={tau_arc:.3f}  ARC-D={tau_arcd:.3f}")

    rows=[]
    for tag,kind,sev in CONDS:
        cc_,kc = conf_correct(cheap,     test_items, kind, sev)
        cd_,kd = conf_correct(distilled, test_items, kind, sev)
        _,  kh = conf_correct(heavy,     test_items, kind, sev)
        keep = cc_>=tau_arc;  karc  = np.where(keep, kc, kh)
        keepd= cd_>=tau_arcd; karcd = np.where(keepd,kd, kh)
        r_arc, r_arcd = 1-keep.mean(), 1-keepd.mean()
        # TOST (ARC-D vs heavy) recorded per condition; headline uses clean
        m,lo,hi = bootstrap_ci(karcd.astype(float)-kh.astype(float))
        base=dict(dataset=DATASET,seed=seed,condition=tag,tost_mean="",tost_lo="",tost_hi="")
        rows += [
            dict(base, system="cheap-B0",     acc=kc.mean(),   gmacs=G_CHEAP, routed="", tau=""),
            dict(base, system="distilled-B0", acc=kd.mean(),   gmacs=G_CHEAP, routed="", tau=""),
            dict(base, system="heavy-Tiny",   acc=kh.mean(),   gmacs=G_HEAVY, routed="", tau=""),
            dict(base, system="ARC",   acc=karc.mean(),  gmacs=G_CHEAP+r_arc*G_HEAVY,  routed=r_arc,  tau=tau_arc),
            dict(base, system="ARC-D", acc=karcd.mean(), gmacs=G_CHEAP+r_arcd*G_HEAVY, routed=r_arcd, tau=tau_arcd,
                 tost_mean=m, tost_lo=lo, tost_hi=hi),
        ]
        print(f"  {tag:8s} ARC-D {karcd.mean()*100:5.1f}% @{G_CHEAP+r_arcd*G_HEAVY:4.2f}GMACs  "
              f"(heavy {kh.mean()*100:5.1f}%)  TOSTd={m*100:+.2f}% [{lo*100:+.2f},{hi*100:+.2f}]")
    append_rows(rows)
    del heavy,cheap,distilled; torch.cuda.empty_cache()
print("done. results ->", RESULTS)

In [ ]:
# ---------- AGGREGATION: run after any/all seeds; prints paper tables ----------
import collections
if not os.path.exists(RESULTS):
    print("no results yet")
else:
    with open(RESULTS) as f: R=list(csv.DictReader(f))
    for ds in sorted({r["dataset"] for r in R}):
        seeds=sorted({int(r["seed"]) for r in R if r["dataset"]==ds})
        print(f"\n===== {ds}  (seeds: {seeds}) =====")
        agg=collections.defaultdict(list)
        for r in R:
            if r["dataset"]!=ds: continue
            agg[(r["system"],r["condition"])].append(float(r["acc"]))
        gm =collections.defaultdict(list)
        for r in R:
            if r["dataset"]!=ds: continue
            gm[(r["system"],r["condition"])].append(float(r["gmacs"]))
        systems=["cheap-B0","distilled-B0","heavy-Tiny","ARC","ARC-D"]
        conds=["clean","jpeg_s2","blur_s2","occ_s1"]
        print(f"{'system':13s} | " + " | ".join(f"{c:>19s}" for c in conds))
        for s in systems:
            cells=[]
            for c in conds:
                a=np.array(agg[(s,c)])*100; g=np.mean(gm[(s,c)])
                cells.append(f"{a.mean():5.1f}±{a.std():4.1f} {g:5.2f}" if len(a)>1 else f"{a.mean():5.1f}      {g:5.2f}")
            print(f"{s:13s} | " + " | ".join(cells))
        # TOST summary (clean, ARC-D vs heavy) per seed
        print("TOST(±1%) clean, ARC-D vs heavy, per seed:")
        for r in R:
            if r["dataset"]==ds and r["system"]=="ARC-D" and r["condition"]=="clean" and r["tost_mean"]!="":
                m,lo,hi=float(r["tost_mean"]),float(r["tost_lo"]),float(r["tost_hi"])
                v="EQUIV" if (lo>-0.01 and hi<0.01) else "n.e."
                print(f"  seed {r['seed']}: {m*100:+.2f}% [{lo*100:+.2f},{hi*100:+.2f}] {v}")

### After the battery completes
Send back: the aggregation output for both datasets + `final_results.csv`. From that I build the consolidated paper tables (two-dataset × five-system × four-condition, mean±std), the TOST statements, and we start drafting. Remaining loose end (parallel, low priority per reviewer): the labeled Kaggle-P100 and CPU rows from the portable hw notebook.